# Compute abundance change for each allele

This script takes in the single-cell FACS data as input, applies various filtering criteria, and then produces a z-score for the abundance of each allele relative to the reference on the same plate.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"

In [ ]:
pDEST_DUAL_df = pd.read_parquet(f"{dualipa_outputs}/facs_single_cell.parquet")

In [ ]:
# Filter out cells with no GFP measurement
pDEST_DUAL_df = pDEST_DUAL_df[pDEST_DUAL_df.GFP > 0]

In [ ]:
for symbol in pDEST_DUAL_df.query('orf_id > 0').symbol.unique():#['MLH1','HBD','GALE','GCK']:#['MLH1','STXBP1','HBD','PMP22','MVK']:
    fig, axs = plt.subplots(2, 2, figsize=(16, 16))
    fig.suptitle(f'{symbol}', fontsize=16)
    for aa_change,ax_x,ax_y in zip(['WT_1','WT_2','WT_3','WT_4'],[0,0,1,1],[0,1,0,1]):
        df = pDEST_DUAL_df.query(f"symbol == '{symbol}' and aa_change == '{aa_change}'")#.sample(n=500)
        if len(df) > 100:
            n_cell = len(df)
            n_valid_cell = len(df.query('mCherry_log10 > 2.98 and mCherry_log10 < 3.98 and GFP_log10 > 2'))
            avg_gfp,avg_mcherry = df.GFP_log10.mean(), df.mCherry_log10.mean()

            x = np.array(df.mCherry_log10)
            y = np.array(df.GFP_log10)
            
            slope, intercept, r_value, p_value, std_err = linregress(x,y)
            #print(slope, intercept, r_value, p_value, std_err)

            sm_x, sm_y = sm_lowess(y, x,  frac=1./5., 
                                       it=5, return_sorted = True).T

            #fig, ax = plt.subplots(figsize=(5, 5))
            axs[ax_x, ax_y].set_title(f'{aa_change} - ncells = {n_cell} / {n_valid_cell}\naverage GFP = {round(np.mean(y),3)} (log10)')
            sns.regplot(data=pDEST_DUAL_df.query("symbol == 'MLH1' and aa_change == 'WT_1'"), 
                        x=df.mCherry_log10, y=np.array(df.GFP_log10),
                        scatter=False,
                        label=f"Lin reg: y={round(slope,2)}x+{round(intercept,2)}",
                        color='royalblue',ax= axs[ax_x, ax_y]
                       )

            axs[ax_x, ax_y].plot(sm_x, sm_y, color='tomato', label='Loess curve')
            axs[ax_x, ax_y].plot(x, y, 'k.', alpha=0.01)
            axs[ax_x, ax_y].hlines(xmin=2.5,xmax=6.5,y=2,linestyles='--',color='darkkhaki', label='GFP threshold')
            axs[ax_x, ax_y].vlines(ymin=2,ymax=6.5,x=3.48-0.5,linestyles='--',color='lightpink', label='mCherry upper window')
            axs[ax_x, ax_y].vlines(ymin=2,ymax=6.5,x=3.48+0.5,linestyles='--',color='lightpink', label='mCherry lower window')

            axs[ax_x, ax_y].set_xlabel('mCherry (log10)')
            axs[ax_x, ax_y].set_ylabel('GFP (log10)')
            axs[ax_x, ax_y].set_xlim(2.5,6.5)
            axs[ax_x, ax_y].set_ylim(0,6.5)
            
            #if [ax_x, ax_y] == [0, 1]:
            #    axs[ax_x, ax_y].legend(bbox_to_anchor=(1.4, 0.8), loc="upper right")
            #    #print()

            axs[ax_x, ax_y].legend()
    plt.savefig(f'figures/WT_GFP_mCherry_scatter/{symbol}.pdf', bbox_inches='tight')
    plt.close(fig)
            
            #summary_l.append([symbol,aa_change,avg_gfp,avg_mcherry,n_cell,n_valid_cell,slope, intercept, r_value, p_value, std_err])
    
    #fig.legend(axs)
    #break
            
#wt_pdest_summary_df = pd.DataFrame(summary_l, columns=['symbol','aa_change','avg_gfp','avg_mcherry','n_cell','n_valid_cell','slope', 'intercept', 'r_value', 'p_value', 'std_err'])
#wt_pdest_summary_df.head()

In [ ]:
# calculate median mCherry for each reference
wt_median_d = dict()
wt_median_l = list()
for orf_id in pDEST_DUAL_df[~pDEST_DUAL_df.orf_id.isnull()].orf_id.unique():
    #print(orf_id)
    #df = pDEST_DUAL_df.query(f"orf_id == {orf_id} and aa_change.str.contains('WT')")
    df = pDEST_DUAL_df.query(f"orf_id == {orf_id} and mut_id == 0")
    for aa_change in df.aa_change.unique():
        sub_df = df.query(f"aa_change == '{aa_change}'")
        n_cell = len(sub_df)
        mdn = sub_df.mCherry_log10.median()
        wt_median_d[(orf_id,aa_change)] = mdn
        wt_median_l.append([orf_id, aa_change, mdn, n_cell])

wt_median_df = pd.DataFrame(wt_median_l, columns=['orf_id', 'aa_change', 'mdn', 'n_cell'])
wt_median_df['100_cells'] = wt_median_df.n_cell >= 100
wt_median_df.head()

In [ ]:
# visualize results for with/without 100 cells
plt.figure(figsize=(40,5))
sns.barplot(data=wt_median_df, x='orf_id',y='mdn', ci=None)
sns.stripplot(data=wt_median_df, x='orf_id',y='mdn', hue='100_cells', palette={True:'black', False:'red'})
plt.xlabel('')
plt.show()

In [ ]:
# I don't know what this means
# Effectively filters mCherry values to only keep between ~970-9700
# We want to filter out mCherry values to only keep those that actually express the gene, but unsure where these values came from
pDEST_DUAL_expr_df = pDEST_DUAL_df[(pDEST_DUAL_df.mCherry_log10 >= (3.487 - 0.5))&(pDEST_DUAL_df.mCherry_log10 <= (3.487 + 0.5))].copy()
pDEST_DUAL_expr_df.head()

In [ ]:
# I don't know what this means
# We filter out variants where we don't get good measurements of the reference
# n_cells_post: number cells that meet criteria of GFP 100, mCherry 800
# Then filter to only keep wells with 200 cells
pDEST_DUAL_expr_WT_df = pDEST_DUAL_expr_df.query("nt_change == 'WT' and GFP_log10 >= 2").copy()
n_cell_d = dict(pDEST_DUAL_expr_WT_df.coordinates.value_counts())
pDEST_DUAL_expr_WT_df['n_cells_post'] = pDEST_DUAL_expr_WT_df.coordinates.map(n_cell_d)
pDEST_DUAL_expr_WT_df['frac_post'] = pDEST_DUAL_expr_WT_df.n_cells_post / pDEST_DUAL_expr_WT_df.n_cells
pDEST_DUAL_expr_WT_df['at_least_200'] = pDEST_DUAL_expr_WT_df.n_cells_post >= 200

pDEST_DUAL_expr_WT_df.head()

In [ ]:
# Apply a GFP threshold here too - maybe it doesn't make sense to filter out those with low GFP? Isn't this removing lots of signal?
# 
df_l = []
for orf_id in pDEST_DUAL_expr_df.orf_id.unique():
    if orf_id > 0:
        
        sub_df = pDEST_DUAL_expr_df.query(f'orf_id == {orf_id} and mut_id == 0 and GFP_log10 >= 2').copy()
        n_cell_d = dict(sub_df.coordinates.value_counts())
        sub_df['n_cells_post'] = sub_df.coordinates.map(n_cell_d)
        sub_df['frac_post'] = sub_df.n_cells_post / sub_df.n_cells
        sub_df = sub_df[(sub_df.frac_post >= 0.237)&(sub_df.n_cells_post>=100)]
        #sub_df['at_least_200'] = sub_df.n_cells_post >= 200
    
        min_wt_n = sub_df.n_cells_post.min()
        wt_l = sub_df.aa_change.unique().tolist()
        
        if min_wt_n >= 100 and len(wt_l) > 1:
            #print(orf_id, min_wt_n, wt_l)
            # Taking a random sample of wt cells?? Equal to the minimum number of cells across variants
            wt_df = pd.concat([sub_df[sub_df.aa_change == wt].sample(n=min_wt_n, random_state=1) for wt in wt_l])
            # Taking many values from pDEST_DUAL_expr_df, which wasn't filtered for 200 cells or 100 cells or GFP value
            allele_df = pDEST_DUAL_expr_df.query(f'orf_id == {orf_id} and mut_id > 0')
            
            df_l.append(wt_df)
            df_l.append(allele_df)
        
reduced_df = pd.concat(df_l)
reduced_df['aa_change_r'] = reduced_df.aa_change.apply(lambda x: x if 'WT' not in x else 'WT')
reduced_df.head()

# reduced_df defined as successes
# unclear what reduced_df is used for

In [ ]:
failed_genes = set(pDEST_DUAL_df.symbol) - set(reduced_df[reduced_df.node_type == 'wt'].symbol)

In [ ]:
# GFP > 100, mCherry > 500, n_cells_postfilter > 800 for 2 out of 4 wild types to consider them good quality
# for the 96 genes with good signal in WT, they profiled the variants

In [ ]:
# calculate allele-wt ratios
res = []
for orf_id in reduced_df.orf_id.unique():
    if orf_id > 0:
        
        sub_df = reduced_df.query(f'orf_id == {orf_id}')
        # mutation id = 0 are WT - change this to explicitly filter on WT for clarity
        wt_gfp_mcherry_ratio = sub_df.query('mut_id == 0').GFP_mCherry_ratio.mean()
        
        info_d = {(orf_id,aa_change_r):(nt_change,clinical_significance) for orf_id,aa_change_r,nt_change,clinical_significance in zip(sub_df.orf_id,
                                                                                                                                       sub_df.aa_change_r,
                                                                                                                                       sub_df.nt_change,
                                                                                                                                       sub_df.clinical_significance
                                                                                                                                       )}
        
        
        for aa_change_r in sub_df.aa_change_r.unique():
            if aa_change_r != 'WT':
                allele_gfp_mcherry_ratio  = sub_df.query(f'aa_change_r == "{aa_change_r}"').GFP_mCherry_ratio.mean()
                res.append([orf_id, 
                            aa_change_r, 
                            info_d[orf_id,aa_change_r][0],
                            info_d[orf_id,aa_change_r][1],
                            wt_gfp_mcherry_ratio, 
                            allele_gfp_mcherry_ratio, 
                            #np.log2((2**allele_gfp_mcherry_ratio)/(2**wt_gfp_mcherry_ratio)), # what math is being done here?
                            allele_gfp_mcherry_ratio - wt_gfp_mcherry_ratio,
                            len(sub_df.query('mut_id == 0')), 
                            len(sub_df.query('aa_change_r == "{aa_change_r}"'))
                           ])
        

allele_wt_ratio_df = pd.DataFrame(res, columns=['orf_id', 'aa_change', 'nt_change', 'clinical_significance', 'wt_gfp_mcherry_ratio' ,'allele_gfp_mcherry_ratio','allele_wt_ratio','n_wt','n_allele'])
allele_wt_ratio_df['symbol'] = allele_wt_ratio_df.orf_id.map(orf_to_symbol)

# removing failed genes
allele_wt_ratio_df = allele_wt_ratio_df.query(f"symbol not in {[e for e in failed_genes if e == e]}").copy()

# calculate mean and std of allele_wt ratio for variants with benign annotations across all genes
# ideally we want to calculate the mean and standard deviation in just the wt of the same gene
# 4 reps not enough, esp b/c sometimes we have less (2 or 3)
# want to estimate the variation of the assay, so are combining data across all benigns
mean_benign_allele_wt_ratio = np.mean(allele_wt_ratio_df.query("clinical_significance == 'Benign'").allele_wt_ratio)
std_benign_allele_wt_ratio = np.std(allele_wt_ratio_df.query("clinical_significance == 'Benign'").allele_wt_ratio)

# z-score defined based on mean and std computed from benigns
allele_wt_ratio_df['z_score'] = allele_wt_ratio_df.allele_wt_ratio.apply(lambda x: (x-mean_benign_allele_wt_ratio)/std_benign_allele_wt_ratio)

# Apply one filter and then another that directly overwrites the results to define unstable vs. stable
# Unclear where this number comes from - one corresponding to p-val 0.05 would make more sense
allele_wt_ratio_df['Stability impact'] = allele_wt_ratio_df.z_score.apply(lambda x: 'Destabilizing' if x <= -1 else 'Stabilizing' if x >=  1 else 'No effect')
allele_wt_ratio_df['Stability impact'] = allele_wt_ratio_df.z_score.apply(lambda x: 'Unstable' if x <= -1 else 'Stable')

allele_wt_ratio_df['experiment_id'] = list(zip(allele_wt_ratio_df.orf_id,allele_wt_ratio_df.nt_change))
allele_wt_ratio_df.head()

In [ ]:
# write output files
reduced_df.to_csv(f'{dualipa_outputs}/pdest_reduced_table.tsv', sep='\t', index=None)
allele_wt_ratio_df.to_csv(f'{dualipa_outputs}/allele_wt_ratio_table.tsv',sep='\t', index=None)